In [1]:
import pandas as pd
from xgboost import XGBClassifier
import lightgbm as lgb
from sklearn.utils import shuffle
import joblib

In [2]:
# conda install pandas
# !pip install -U pandas --break-system-packages
# !pip install -U lightgbm --break-system-packages
# !pip uninstall lightgbm --break-system-packages
# !pip install joblib --break-system-packages
# !pip install optuna --break-system-packages

# !pip install pyopencl --break-system-packages

In [3]:
lst=[['male', 'female','zy','mz','age','mth','wk','wbc','neu','lym','mon','eos', 'rbc', 'hgb', 'mcv', 'rdwsd', 'rdwcv', 'hct', 'plt', 'pdw', 'pct', 'plcr', 'label'],
    ['male', 'female','zy', 'mz', 'age', 'wbc', 'neu', 'lym', 'mon','eos', 'rbc', 'hgb', 'mcv', 'rdwsd', 'rdwcv', 'hct','plt','label'],
    ['age', 'wbc', 'neu', 'lym', 'mon','eos', 'rbc', 'hgb', 'mcv', 'rdwsd', 'rdwcv', 'hct','plt','label'],
    ['age', 'wbc', 'neu', 'lym', 'mon','eos', 'rbc', 'hgb', 'mcv', 'hct','plt','label']]
fnams=['all','tim_plt3','tim_plt3_sex_zy_mz','tim_plt3_sex_zy_mz_sd_cv']

In [4]:
Dc=pd.read_csv('./data/cg_train_2019_to_2024-09_329692.csv')

/tmp/ipykernel_15896/1656769690.py:1: DtypeWarning: Columns (31) have mixed types. Specify dtype option on import or set low_memory=False.
  Dc=pd.read_csv('./data/cg_train_2019_to_2024-09_329692.csv')


In [5]:
import optuna
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier

def objective(trial):
    params = {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "booster": "gbtree",
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-5, 10, log=True),
        # "random_state": 42,
        "n_jobs": -1,
        "verbosity": 0,
        "device":"cuda",
        "tree_method":"hist"
    }

    model = XGBClassifier(**params, n_estimators=500)
    scores = cross_val_score(model, x_train, y_train, cv=3, scoring='neg_log_loss')
    return scores.mean()  # higher = better (less negative)

# Optimize
# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=50)

# print("Best params:", study.best_params)

In [6]:
# xgboost
# time_n=0
D_list_ten=Dc
for n in range(1,2):
    for k in range(10):

        D_train=D_list_ten[lst[n]]
        D_train=shuffle(D_train)
        D_train=D_train.reset_index(drop=True)
        dtrain=D_train.sample(frac=0.8,replace=False)
        dval=D_train.drop(index=dtrain.index)
        x_train,y_train=dtrain.drop('label',axis=1),dtrain['label']
        x_val,y_val=dval.drop('label',axis=1),dval['label']

        # Optimize
        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=50)
        best_params = study.best_params
        best_params.update({
            "n_estimators": 10000,
            # "random_state": 42,
            "n_jobs": -1
        })
        
        xgb_model = XGBClassifier(early_stopping_rounds=50,**best_params)
        xgb_model.fit(
            x_train, y_train,
            eval_set=[(x_val,y_val)],
            verbose=False
        )
        joblib.dump(xgb_model,"./models_xgb_lgb/tune_xgb_train329692_"+fnams[n]+'_k'+str(k)+'.json')
        

[I 2025-10-31 00:12:37,079] A new study created in memory with name: no-name-b8433af1-eb3a-4d85-a9b5-ef2e1c12d31d
[I 2025-10-31 00:12:39,254] Trial 0 finished with value: -0.4737577522197218 and parameters: {'max_depth': 5, 'learning_rate': 0.02788943896904504, 'subsample': 0.7455971237741662, 'colsample_bytree': 0.8629382990043339, 'min_child_weight': 7, 'gamma': 0.5233151057217161, 'reg_alpha': 0.09467568987209175, 'reg_lambda': 2.2477438678641527e-05}. Best is trial 0 with value: -0.4737577522197218.
[I 2025-10-31 00:12:47,063] Trial 1 finished with value: -0.46925381866723237 and parameters: {'max_depth': 9, 'learning_rate': 0.010057778631407088, 'subsample': 0.8914849218538199, 'colsample_bytree': 0.7755368061436242, 'min_child_weight': 1, 'gamma': 1.2098692446435144, 'reg_alpha': 0.00013625927590166454, 'reg_lambda': 0.00358192421473138}. Best is trial 1 with value: -0.46925381866723237.
[I 2025-10-31 00:12:48,231] Trial 2 finished with value: -0.47662977472920315 and parameters:

In [7]:
import lightgbm as lgb
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.metrics import log_loss
import numpy as np
import pandas as pd

def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'boosting_type': 'gbdt',
        'verbosity': -1,
        # 'random_state': 42,
        'device': 'gpu',
        'gpu_use_dp': False,      # single precision = faster on GPU
        'num_leaves': trial.suggest_int('num_leaves', 20, 128),       
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.2, log=True),  
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 200),      
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),     
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        # Optional: limit CPU threads (GPU does most work)
        'num_threads': 4,         # just for data loading/preprocessing
    }
    cv = StratifiedKFold(n_splits=3, shuffle=True)
    logloss_scores = []
    
    X, y=x_train,y_train
    X = X.astype('float32')
    y = y.astype('int32')
    
    for train_idx, val_idx in cv.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        train_ds = lgb.Dataset(X_tr, y_tr)
        val_ds = lgb.Dataset(X_val, y_val, reference=train_ds)

        model = lgb.train(
            params,
            train_ds,
            valid_sets=[val_ds],
            num_boost_round=10000,
            callbacks=[
                lgb.early_stopping(stopping_rounds=50, verbose=False),
                lgb.log_evaluation(False)
            ]
        )

        y_pred_proba = model.predict(X_val, num_iteration=model.best_iteration)
        # Clip predictions to avoid log(0) — sklearn does this internally, but safe to do
        y_pred_proba = np.clip(y_pred_proba, 1e-15, 1 - 1e-15)
        loss = log_loss(y_val, y_pred_proba)
        logloss_scores.append(loss)

    return np.mean(logloss_scores)  # Optuna will MINIMIZE this

In [ ]:
Dc_lgb=pd.read_csv('./data/cg_train_2019_to_2024-09_329692.csv')

In [8]:
# lightgbm
# Train the model

D_list_ten=Dc_lgb
for n in range(1,2):
    for k in range(10):
        D_train=D_list_ten[lst[n]]  
        D_train=shuffle(D_train)
        D_train=D_train.reset_index(drop=True)
        dtrain=D_train.sample(frac=0.8,replace=False)
        dval=D_train.drop(index=dtrain.index)
        
        x_train,y_train=dtrain.drop('label',axis=1),dtrain['label']
        x_val,y_val=dval.drop('label',axis=1),dval['label']          

        train_ds = lgb.Dataset(x_train, y_train)
        val_ds = lgb.Dataset(x_val, y_val)

        # Run optimization
                    
        # Create study to MINIMIZE log loss
        study = optuna.create_study(direction='minimize')
        study.optimize(objective, n_trials=50)
        
        # Train the model
        best_params = study.best_params
        best_params.update({
            'max_bin': 512,           # now use higher binning
            'num_threads': -1,        # use all cores
            'feature_pre_filter': True,
            "learning_rate": 0.025,
            "boosting_type": "gbdt",
        })
       
        # Optional: increase `num_leaves` slightly and lower `learning_rate` for final model
        # (e.g., halve learning_rate and double expected n_estimators)
        bst  = lgb.train(
            best_params,
            train_ds,
            valid_sets=[val_ds],
            num_boost_round=10000,
            callbacks=[
                lgb.early_stopping(stopping_rounds=50),
                lgb.log_evaluation(10)
            ]
        )
        # Save the model as JSON
        joblib.dump(bst, "./models_xgb_lgb/tune_lgb_train329692_"+fnams[n]+'_k'+str(k)+'.json')

/tmp/ipykernel_200420/514608140.py:11: DtypeWarning: Columns (31) have mixed types. Specify dtype option on import or set low_memory=False.
  Dc=pd.read_csv('./data/cg_train_2019_to_'+lst_cg_train_2019_to_[i])
[I 2025-10-30 07:47:33,220] A new study created in memory with name: no-name-788c8425-6624-4bc6-a422-c8ad2ef88df9
[I 2025-10-30 07:47:36,792] Trial 0 finished with value: 0.4558133109291447 and parameters: {'num_leaves': 34, 'learning_rate': 0.04829714797460554, 'min_child_samples': 95, 'reg_lambda': 0.002661284895568417, 'subsample': 0.9934511806903463, 'colsample_bytree': 0.6559933223419928}. Best is trial 0 with value: 0.4558133109291447.
[I 2025-10-30 07:47:39,204] Trial 1 finished with value: 0.45554916511508176 and parameters: {'num_leaves': 98, 'learning_rate': 0.09346794711957038, 'min_child_samples': 60, 'reg_lambda': 2.13586596554105e-05, 'subsample': 0.9477521883884759, 'colsample_bytree': 0.6351862871471221}. Best is trial 1 with value: 0.45554916511508176.
[I 2025-10

Training until validation scores don't improve for 30 rounds
[10]	valid_0's l2: 0.215065
[20]	valid_0's l2: 0.192563
[30]	valid_0's l2: 0.178422
[40]	valid_0's l2: 0.168023
[50]	valid_0's l2: 0.161291
[60]	valid_0's l2: 0.156801
[70]	valid_0's l2: 0.153499
[80]	valid_0's l2: 0.151335
[90]	valid_0's l2: 0.149708
[100]	valid_0's l2: 0.148547
[110]	valid_0's l2: 0.14763
[120]	valid_0's l2: 0.146962
[130]	valid_0's l2: 0.14644
[140]	valid_0's l2: 0.146046
[150]	valid_0's l2: 0.14571
[160]	valid_0's l2: 0.145398
[170]	valid_0's l2: 0.145148
[180]	valid_0's l2: 0.144966
[190]	valid_0's l2: 0.1448
[200]	valid_0's l2: 0.144634
[210]	valid_0's l2: 0.144523
[220]	valid_0's l2: 0.144433
[230]	valid_0's l2: 0.144331
[240]	valid_0's l2: 0.144227
[250]	valid_0's l2: 0.144156
[260]	valid_0's l2: 0.144115
[270]	valid_0's l2: 0.144061
[280]	valid_0's l2: 0.143995
[290]	valid_0's l2: 0.14393
[300]	valid_0's l2: 0.143907
[310]	valid_0's l2: 0.143863
[320]	valid_0's l2: 0.143835
[330]	valid_0's l2: 0.1437

[I 2025-10-30 07:52:04,160] A new study created in memory with name: no-name-7a592d0f-7ff6-4761-a2a5-1c2f5995ebb0
[I 2025-10-30 07:52:07,705] Trial 0 finished with value: 0.4525538142295306 and parameters: {'num_leaves': 58, 'learning_rate': 0.04953463593266601, 'min_child_samples': 73, 'reg_lambda': 0.21744792839809704, 'subsample': 0.7215018911350655, 'colsample_bytree': 0.6187293602786402}. Best is trial 0 with value: 0.4525538142295306.
[I 2025-10-30 07:52:09,514] Trial 1 finished with value: 0.4533521442416593 and parameters: {'num_leaves': 40, 'learning_rate': 0.10309113627710353, 'min_child_samples': 149, 'reg_lambda': 0.0006839124589227772, 'subsample': 0.6648964818264386, 'colsample_bytree': 0.8022327009483793}. Best is trial 0 with value: 0.4525538142295306.
[I 2025-10-30 07:52:10,872] Trial 2 finished with value: 0.45595484871243436 and parameters: {'num_leaves': 38, 'learning_rate': 0.13104099678975542, 'min_child_samples': 11, 'reg_lambda': 1.0113766099294547e-06, 'subsamp

Training until validation scores don't improve for 30 rounds
[10]	valid_0's l2: 0.216488
[20]	valid_0's l2: 0.194401
[30]	valid_0's l2: 0.180637
[40]	valid_0's l2: 0.17046
[50]	valid_0's l2: 0.163843
[60]	valid_0's l2: 0.159328
[70]	valid_0's l2: 0.15607
[80]	valid_0's l2: 0.153937
[90]	valid_0's l2: 0.152345
[100]	valid_0's l2: 0.151182
[110]	valid_0's l2: 0.150265
[120]	valid_0's l2: 0.149529
[130]	valid_0's l2: 0.149028
[140]	valid_0's l2: 0.148601
[150]	valid_0's l2: 0.148258
[160]	valid_0's l2: 0.147968
[170]	valid_0's l2: 0.147726
[180]	valid_0's l2: 0.147472
[190]	valid_0's l2: 0.147244
[200]	valid_0's l2: 0.14706
[210]	valid_0's l2: 0.146924
[220]	valid_0's l2: 0.146827
[230]	valid_0's l2: 0.14675
[240]	valid_0's l2: 0.146628
[250]	valid_0's l2: 0.146567
[260]	valid_0's l2: 0.146514
[270]	valid_0's l2: 0.146464
[280]	valid_0's l2: 0.146405
[290]	valid_0's l2: 0.146373
[300]	valid_0's l2: 0.146337
[310]	valid_0's l2: 0.146274
[320]	valid_0's l2: 0.146232
[330]	valid_0's l2: 0.14

[I 2025-10-30 07:56:45,761] A new study created in memory with name: no-name-1f879ed7-52be-4191-a8d0-2856e40965df
[I 2025-10-30 07:56:47,905] Trial 0 finished with value: 0.45635195638853115 and parameters: {'num_leaves': 27, 'learning_rate': 0.07955291077273996, 'min_child_samples': 17, 'reg_lambda': 0.00029372984079095363, 'subsample': 0.6219564340214322, 'colsample_bytree': 0.9188573416388089}. Best is trial 0 with value: 0.45635195638853115.
[I 2025-10-30 07:56:50,380] Trial 1 finished with value: 0.4548921012811796 and parameters: {'num_leaves': 111, 'learning_rate': 0.08381586867310978, 'min_child_samples': 64, 'reg_lambda': 0.016872816438190278, 'subsample': 0.7135628777530901, 'colsample_bytree': 0.8151916533703755}. Best is trial 1 with value: 0.4548921012811796.
[I 2025-10-30 07:56:54,069] Trial 2 finished with value: 0.4542708407505434 and parameters: {'num_leaves': 73, 'learning_rate': 0.041294536306336116, 'min_child_samples': 195, 'reg_lambda': 1.441079909781846e-06, 'sub

...

Training until validation scores don't improve for 30 rounds
[10]	valid_0's l2: 0.218023
[20]	valid_0's l2: 0.1973
[30]	valid_0's l2: 0.184548
[40]	valid_0's l2: 0.175142
[50]	valid_0's l2: 0.168928
[60]	valid_0's l2: 0.164745
[70]	valid_0's l2: 0.161716
[80]	valid_0's l2: 0.159695
[90]	valid_0's l2: 0.158095
[100]	valid_0's l2: 0.156994
[110]	valid_0's l2: 0.156143
[120]	valid_0's l2: 0.15547
[130]	valid_0's l2: 0.154969
[140]	valid_0's l2: 0.154546
[150]	valid_0's l2: 0.154186
[160]	valid_0's l2: 0.153898
[170]	valid_0's l2: 0.153629
[180]	valid_0's l2: 0.153408
[190]	valid_0's l2: 0.153231
[200]	valid_0's l2: 0.15305
[210]	valid_0's l2: 0.152897
[220]	valid_0's l2: 0.152732
[230]	valid_0's l2: 0.15263
[240]	valid_0's l2: 0.15251
[250]	valid_0's l2: 0.152401
[260]	valid_0's l2: 0.152328
[270]	valid_0's l2: 0.152251
[280]	valid_0's l2: 0.15221
[290]	valid_0's l2: 0.152141
[300]	valid_0's l2: 0.152096
[310]	valid_0's l2: 0.152052
[320]	valid_0's l2: 0.152007
[330]	valid_0's l2: 0.15195

[I 2025-10-30 14:28:19,336] A new study created in memory with name: no-name-78755751-86b7-4712-b4f3-fc4691a0b19a
[I 2025-10-30 14:28:27,014] Trial 0 finished with value: 0.4680838016139448 and parameters: {'num_leaves': 43, 'learning_rate': 0.05434263152809035, 'min_child_samples': 69, 'reg_lambda': 1.1734606931482497e-06, 'subsample': 0.948230937369185, 'colsample_bytree': 0.902714749165219}. Best is trial 0 with value: 0.4680838016139448.
[I 2025-10-30 14:28:33,291] Trial 1 finished with value: 0.46818974726786844 and parameters: {'num_leaves': 61, 'learning_rate': 0.07780593027157781, 'min_child_samples': 42, 'reg_lambda': 9.95237922885424e-06, 'subsample': 0.8845809219053127, 'colsample_bytree': 0.6002379844209432}. Best is trial 0 with value: 0.4680838016139448.
[I 2025-10-30 14:28:41,409] Trial 2 finished with value: 0.4685896200874469 and parameters: {'num_leaves': 41, 'learning_rate': 0.037553890912456936, 'min_child_samples': 200, 'reg_lambda': 0.0020848994874978727, 'subsamp

Training until validation scores don't improve for 30 rounds
[10]	valid_0's l2: 0.21788
[20]	valid_0's l2: 0.19703
[30]	valid_0's l2: 0.184117
[40]	valid_0's l2: 0.174559
[50]	valid_0's l2: 0.168256
[60]	valid_0's l2: 0.16396
[70]	valid_0's l2: 0.160818
[80]	valid_0's l2: 0.158722
[90]	valid_0's l2: 0.157059
[100]	valid_0's l2: 0.155893
[110]	valid_0's l2: 0.154956
[120]	valid_0's l2: 0.154198
[130]	valid_0's l2: 0.153673
[140]	valid_0's l2: 0.153213
[150]	valid_0's l2: 0.152827
[160]	valid_0's l2: 0.152509
[170]	valid_0's l2: 0.152238
[180]	valid_0's l2: 0.151983
[190]	valid_0's l2: 0.151782
[200]	valid_0's l2: 0.151598
[210]	valid_0's l2: 0.151445
[220]	valid_0's l2: 0.151299
[230]	valid_0's l2: 0.15116
[240]	valid_0's l2: 0.151049
[250]	valid_0's l2: 0.150946
[260]	valid_0's l2: 0.150871
[270]	valid_0's l2: 0.150813
[280]	valid_0's l2: 0.150754
[290]	valid_0's l2: 0.15068
[300]	valid_0's l2: 0.150634
[310]	valid_0's l2: 0.15058
[320]	valid_0's l2: 0.150523
[330]	valid_0's l2: 0.1504

[I 2025-10-30 14:37:47,053] A new study created in memory with name: no-name-f593dd9b-8ff6-44d3-a28a-f11b30ec8887
[I 2025-10-30 14:37:50,343] Trial 0 finished with value: 0.47156419837551483 and parameters: {'num_leaves': 20, 'learning_rate': 0.19495291337949286, 'min_child_samples': 21, 'reg_lambda': 0.8987899254546742, 'subsample': 0.7580787041429973, 'colsample_bytree': 0.6091709518793701}. Best is trial 0 with value: 0.47156419837551483.
[I 2025-10-30 14:37:57,798] Trial 1 finished with value: 0.4671840694451594 and parameters: {'num_leaves': 123, 'learning_rate': 0.0592534259731205, 'min_child_samples': 193, 'reg_lambda': 0.003927359088513155, 'subsample': 0.7986651636315978, 'colsample_bytree': 0.802844305721409}. Best is trial 1 with value: 0.4671840694451594.
[I 2025-10-30 14:38:00,802] Trial 2 finished with value: 0.4696933066081565 and parameters: {'num_leaves': 117, 'learning_rate': 0.15747731788956226, 'min_child_samples': 150, 'reg_lambda': 3.04985973181932e-08, 'subsample

Training until validation scores don't improve for 30 rounds
[10]	valid_0's l2: 0.218856
[20]	valid_0's l2: 0.197738
[30]	valid_0's l2: 0.184706
[40]	valid_0's l2: 0.175056
[50]	valid_0's l2: 0.168823
[60]	valid_0's l2: 0.16438
[70]	valid_0's l2: 0.16123
[80]	valid_0's l2: 0.159181
[90]	valid_0's l2: 0.157481
[100]	valid_0's l2: 0.156298
[110]	valid_0's l2: 0.155359
[120]	valid_0's l2: 0.154575
[130]	valid_0's l2: 0.154067
[140]	valid_0's l2: 0.153598
[150]	valid_0's l2: 0.153204
[160]	valid_0's l2: 0.152899
[170]	valid_0's l2: 0.152595
[180]	valid_0's l2: 0.152334
[190]	valid_0's l2: 0.152127
[200]	valid_0's l2: 0.151926
[210]	valid_0's l2: 0.151757
[220]	valid_0's l2: 0.151597
[230]	valid_0's l2: 0.151464
[240]	valid_0's l2: 0.151328
[250]	valid_0's l2: 0.151203
[260]	valid_0's l2: 0.151079
[270]	valid_0's l2: 0.151002
[280]	valid_0's l2: 0.150926
[290]	valid_0's l2: 0.150852
[300]	valid_0's l2: 0.150794
[310]	valid_0's l2: 0.150737
[320]	valid_0's l2: 0.150682
[330]	valid_0's l2: 0.